## Imports

In [1]:
import math
import os
import time
import requests
import pandas as pd
import json
import numpy as np
from pathlib import Path
from typing import Optional, Dict, List, Union
from unidecode import unidecode
import matplotlib.pyplot as plt

import pandas_gbq
from google.auth import default
from google.cloud import bigquery
from google.api_core.exceptions import NotFound

import warnings
warnings.filterwarnings("ignore")

/Users/lucas.nascimento/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/lucas.nascimento/Library/Python/3.9/lib/python/site-packages/google/api_core/_python_version_support.py:242: FutureWarning: You are using a non-supported Python version (3.9.6). Google will not post any further updates to google.api_core supporting this Python version. Please upgrade to the latest Python version, or at least Python 3.10, and then update google.api_core.
  warnings.warn(message, FutureWarning)
/Users/lucas.nascimento/Library/Python/3.9/lib/python/site-packages/google/auth/__init__.py:54: FutureWarning: You are using a Python version 3.9 past its end of life. Google will update google-auth with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please u

In [2]:
from funcoes_escoragem import *

## Diretório

In [3]:
BASE_DIR = Path("data")
RAW_DIR = BASE_DIR / "raw"
TRUSTED_DIR = BASE_DIR / "trusted"
ANALYTICS_DIR = BASE_DIR / "analytics"

for path in [RAW_DIR, TRUSTED_DIR, ANALYTICS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

## Google BigQuery

In [4]:
project_id = 'loft-dl-fintech'

# CredPago - Consulta Realizada**

In [5]:
query_credpago = f"""
WITH consulta_realizada AS (
    SELECT
        CAST(REGEXP_REPLACE(documento, r'[^0-9]', '') AS INT64) AS CPF_CNPJ,
        id_externo AS contract_id,

        MIN(DATE(data)) OVER (
            PARTITION BY id_externo
        ) AS requested_at,

        MAX(DATE(data)) OVER (
            PARTITION BY id_externo
        ) AS data_ultima_consulta,
        ARRAY_LENGTH(JSON_EXTRACT_ARRAY(CR.json_retornado, '$.pessoas')) AS qtd_proponentes,
        CR.*
    FROM `loft-dl-fintech.bronze_credpago_sortinghat.consulta_realizada` CR
    WHERE DATE(data) >= DATE_SUB(CURRENT_DATE(), INTERVAL 4 WEEK)
    AND DATE(data) <= CURRENT_DATE()
)

SELECT *
FROM consulta_realizada
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY contract_id
    ORDER BY 
        CASE WHEN model = 'BLEND_4' THEN 1 ELSE 2 END ASC,
        data DESC
) = 1
"""

credpago_df = pd.read_gbq(query_credpago, project_id=project_id)
credpago_df

,CPF_CNPJ,contract_id,requested_at,data_ultima_consulta,qtd_proponentes,score_imobiliaria,renda_considerada,model,id_externo,json_retornado,...,id_funcionalidade,_sdc_table_version,request,_sdc_received_at,_sdc_sequence,documento,rating,_sdc_batched_at,data,result
0,33105054804,4152079,2026-06-18,2026-06-18,1,NI,6850.000000000,BLEND3_3,4152079,"{""pessoas"":[{""nome"":""CRISTIANO CARDOSO DA SILV...",...,32,1778785248777,"{""valor_aluguel"":""1212"",""imobiliaria_id"":44123...",2026-06-19 08:00:36+00:00,1781856034797109435,33105054804,B,2026-06-19 08:01:41.152000+00:00,2026-06-18 19:36:37+00:00,APROVAR
1,7691151527,4156419,2026-06-16,2026-06-16,1,NI,3288.000000000,BLEND3_3,4156419,"{""pessoas"":[{""nome"":""JOAO VITOR GOIS"",""documen...",...,34,1778785248777,"{""valor_aluguel"":""1000"",""imobiliaria_id"":55442...",2026-06-17 08:00:35+00:00,1781683231356495457,07691151527,C,2026-06-17 08:04:49.776000+00:00,2026-06-16 15:29:49+00:00,APROVAR
2,40664954863,4159085,2026-06-18,2026-06-18,1,E,1849.500000000,BLEND3_3,4159085,"{""pessoas"":[{""nome"":""JONATHAN RAMOS MARCOS"",""d...",...,34,1778785248777,"{""valor_aluguel"":""1350"",""imobiliaria_id"":3037,...",2026-06-18 18:00:35+00:00,1781805631317964210,40664954863,C,2026-06-18 18:10:18.202000+00:00,2026-06-18 11:32:48+00:00,APROVAR
3,12490765957,4165665,2026-06-12,2026-06-20,1,NI,1918.000000000,BLEND3_3,4165665,"{""pessoas"":[{""nome"":""VINICIUS SANTOS FIRMINO"",...",...,34,1778785248777,"{""valor_aluguel"":""10000"",""imobiliaria_id"":5364...",2026-06-20 18:00:25+00:00,1781978422644977544,12490765957,C,2026-06-20 18:05:10.481000+00:00,2026-06-20 11:37:31+00:00,DERIVAR
4,12010654536,4167478,2026-06-15,2026-06-15,1,NI,1575.500000000,BLEND3_3,4167478,"{""pessoas"":[{""nome"":""WILLIANY STEFANY LIMA MOT...",...,34,1778785248777,"{""valor_aluguel"":""1000"",""imobiliaria_id"":55442...",2026-06-16 08:00:47+00:00,1781596843109640522,12010654536,C,2026-06-16 08:04:56.462000+00:00,2026-06-15 15:55:12+00:00,APROVAR
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
111990,8227376983,4345686,2026-07-07,2026-07-07,1,D,1575.500000000,BLEND3_3,4345686,"{""pessoas"":[{""nome"":""FLAVIA DOS SANTOS ELIAS"",...",...,33,1778785248777,"{""valor_aluguel"":2500,""matchmaking_on"":false,""...",2026-07-08 08:00:27+00:00,1783497625603264832,08227376983,E,2026-07-08 08:02:00.285000+00:00,2026-07-07 18:05:05+00:00,REPROVAR
111991,48788388840,4345766,2026-07-07,2026-07-07,1,F,1644.000000000,BLEND3_3,4345766,"{""pessoas"":[{""nome"":""JEAN VENICIOS DE LARA"",""d...",...,33,1778785248777,"{""valor_aluguel"":1037.19,""matchmaking_on"":fals...",2026-07-08 08:00:27+00:00,1783497625732630529,48788388840,E,2026-07-08 08:02:00.315000+00:00,2026-07-07 18:29:02+00:00,REPROVAR
111992,47313079885,4345814,2026-07-07,2026-07-07,1,A,1712.500000000,BLEND3_3,4345814,"{""pessoas"":[{""nome"":""MILENA SILVA DE ARAUJO"",""...",...,32,1778785248777,"{""valor_aluguel"":1700,""matchmaking_on"":false,""...",2026-07-08 08:00:27+00:00,1783497626033028096,47313079885,B,2026-07-08 08:02:00.380000+00:00,2026-07-07 20:50:38+00:00,APROVAR
111993,62043840312,4345874,2026-07-07,2026-07-07,1,NI,2740.000000000,BLEND3_3,4345874,"{""pessoas"":[{""nome"":""SANDERSON NUNES BARROS"",""...",...,34,1778785248777,"{""valor_aluguel"":1460,""matchmaking_on"":false,""...",2026-07-08 08:00:27+00:00,1783497625919867358,62043840312,C,2026-07-08 08:02:00.358000+00:00,2026-07-07 19:32:22+00:00,APROVAR


In [6]:
mask = credpago_df['contract_id'].astype(str).str.match(r'^\d+$')
credpago_df = credpago_df[mask].copy()
credpago_df['contract_id'] = credpago_df['contract_id'].astype(int)

In [7]:
credpago_df['model'].value_counts(dropna=False)

model
BLEND3_3                95712
BLEND_REGRESSAO_2026     7799
HVA3                     4475
BVS_CUSTOM               1628
BLEND_4                  1613
HVA4                      712
BVS_CUSTOM_V2              39
                           11
HFT1                        6
Name: count, dtype: int64

## Multiproponente

In [8]:
credpago_df['qtd_proponentes'].value_counts(dropna=False)

qtd_proponentes
1    107611
2      4046
3       306
4        30
6         1
5         1
Name: count, dtype: Int64

In [9]:
credpago_df['qtd_proponentes'].value_counts(normalize=True, dropna=False)

qtd_proponentes
1    0.960855
2    0.036127
3    0.002732
4    0.000268
6    0.000009
5    0.000009
Name: proportion, dtype: Float64

## Quebrar JSON - Retornado

In [10]:
def unwrap_payload(obj):
    """Supports old format (message wrapper) and new format (root payload)."""
    if not obj:
        return None
    if isinstance(obj, dict) and "message" in obj:
        return obj["message"]
    return obj

def get_pessoas(obj):
    payload = unwrap_payload(obj)
    return (payload or {}).get("pessoas") or []

In [11]:
parsed = credpago_df["json_retornado"].apply(
    lambda x: json.loads(x) if pd.notna(x) and x else None
)

valid_parsed = parsed.dropna()
payload_parsed = valid_parsed.apply(unwrap_payload)

contrato_df = (
    pd.json_normalize(payload_parsed, sep="_")
    .add_prefix("message_")  # keeps message_decisao, message_scores_*, etc.
)

pessoa_df = pd.json_normalize(
    payload_parsed.apply(lambda x: (get_pessoas(x) or [{}])[0]),
    sep="_",
).add_prefix("pessoa_")

In [12]:
valid = parsed.notna()
base_idx = credpago_df.loc[valid].index

pendencias = []
for idx, row in parsed[valid].items():
    for p in get_pessoas(row):
        if not isinstance(p, dict):
            continue

        serasa = p.get("anotacoesFinanceirasSerasa") or {}
        pend_list = serasa.get("PendenciasPagamentoPF") or []

        for pend in pend_list:
            if isinstance(pend, dict):
                pendencias.append({"idx": idx, **pend})

pendencias_df = pd.DataFrame(pendencias)

if not pendencias_df.empty:
    pendencias_agg = (
        pendencias_df
        .groupby("idx", as_index=False)
        .agg(
            qtde_pefin=("Valor", "count"),
            valor_pefin_total=("Valor", lambda s: pd.to_numeric(s, errors="coerce").sum()),
            modalidades_pefin=("Modalidade", lambda s: ",".join(sorted(set(s.dropna().astype(str))))),
        )
    )
else:
    pendencias_agg = pd.DataFrame(columns=["idx", "qtde_pefin", "valor_pefin_total", "modalidades_pefin"])

## Quebrar JSON - Request


In [13]:
request_parsed = credpago_df["request"].apply(
    lambda x: json.loads(x) if pd.notna(x) and x else None
)

request_df = (
    pd.json_normalize(request_parsed.dropna(), sep="_")
    .add_prefix("request_")
)

# Unify old (snake_case) and new (camelCase) request schemas
REQUEST_FIELD_ALIASES = {
    "request_valorAluguel": ["request_valor_aluguel"],
    "request_imobiliariaId": ["request_imobiliaria_id"],
    "request_idExterno": ["request_id_externo"],
    "request_imovelTipo": ["request_imovel_tipo"],
    "request_principalDocument": ["request_pessoa_principal_documento"],
}

for target, sources in REQUEST_FIELD_ALIASES.items():
    existing_sources = [c for c in sources if c in request_df.columns]
    if not existing_sources and target not in request_df.columns:
        continue
    if target not in request_df.columns:
        request_df[target] = pd.NA
    for source in existing_sources:
        if target == "request_valorAluguel":
            request_df[target] = request_df[target].combine_first(
                pd.to_numeric(request_df[source], errors="coerce")
            )
        else:
            request_df[target] = request_df[target].combine_first(request_df[source])
        request_df = request_df.drop(columns=[source])


## Join Json's

In [14]:
EXPANDED_PREFIXES = ("message_", "pessoa_", "request_")
expanded_cols = [c for c in credpago_df.columns if c.startswith(EXPANDED_PREFIXES)]

base_df = credpago_df.loc[valid].drop(columns=expanded_cols, errors="ignore")

credpago_expandido = (
    base_df
    .join(contrato_df, how="left")
    .join(pessoa_df, how="left")
    .join(request_df, how="left")
    .reset_index(names="idx")
    .merge(pendencias_agg, on="idx", how="left")
    .drop(columns="idx")
)


In [15]:
cols_drop = [
    # ingestão
    "_sdc_table_version", "_sdc_received_at", "_sdc_sequence", "_sdc_batched_at",

    # json bruto / containers
    "json_retornado",
    "request",
    "message_pessoas", "message_scores", "message_ratings",
    "pessoa_scores", "pessoa_ratings", "message_blend3Variables",
    "request_priorDecisions", "request_dadosAusentes", "request_errosTecnicos",
    "request_outras_pessoas", "request_blend3Variables", "request_scores",

    # bronze redundante
    "score_bvs",
    "renda_considerada",
    "decisao_bureau",
    "limite_mensal",

    # prováveis duplicatas
    "id_externo",
    "data",
    "request_month",

    # operacional / baixo valor (old format only — safe to keep)
    "success", "message_manual", "message_node",
    "message_reaproveitamentoConsultaMotor", "message_savingBureauPowerCurve",
    "message_logs", "message_partnerIds",
    "message_errosTecnicos", "message_dadosAusentes",
    "pessoa_errosTecnicos", "pessoa_dadosAusentes",
    "message_rentGuaranteeConstraints_rent_coverage_multiplier",
    "message_rentGuaranteeConstraints_exit_cost_multiplier",
    "message_rentGuaranteeConstraints_commission_percent",
    "message_rentGuaranteeConstraints_version",
]

credpago_clean = credpago_expandido.drop(columns=[c for c in cols_drop if c in credpago_expandido.columns])


In [16]:
credpago_clean[
    (pd.to_datetime(credpago_clean["requested_at"]).dt.normalize() == "2026-07-03")
    & (credpago_clean["message_decisao"] == "BLEND_4")
][["message_blend3Variables_realEstatePc4HistoryOver100Contracts", "message_blend3Variables_cityPc4HistoryOver100Contracts"]]

,message_blend3Variables_realEstatePc4HistoryOver100Contracts,message_blend3Variables_cityPc4HistoryOver100Contracts
950,0.000000,0.104675
952,0.000000,0.130864
2049,0.000000,0.000000
2051,0.000000,0.140461
3123,0.159259,0.107708
...,...,...
110754,0.000000,0.076716
110761,0.000000,0.087567
110766,NaN,0.130864
111848,0.000000,0.058575


## Escoragem Blend4

In [17]:
cp_final_df = credpago_clean.replace({
    "request_imovelTipo": {"RESIDENCIAL": 1, "COMERCIAL": 0}, 
    "message_blend3Variables_hasPreviousContracts": {True: 1, False: 0},
    "message_blend3Variables_hadOverdueInvoiceInPreviousContracts": {True: 1, False: 0}
}
)

In [18]:
rename_dict = {
    "message_scores_BVS_CUSTOM_V2": "score_proposto__bvs",#
    "message_scores_HFT1": "SERASA_CHSV5",
    "pessoa_idade": "age",
    "request_imovelTipo": "property_type",
    "message_qtdeRestricoesContrato": "qtde_restricoes__consulta_realizada",
    "request_valorAluguel": "rental_value",
    "message_rendaConsideradaContrato": "income",
    "message_blend3Variables_realEstatePc4HistoryOver100Contracts": "agency_pc4_mais_100_contratos__pc_categorias",
    "message_blend3Variables_cityPc4HistoryOver100Contracts": "city_pc4_mais_100_contratos__pc_categorias",
    "message_blend3Variables_hasPreviousContracts": "flag_tem__contratos_anteriores",
    "message_blend3Variables_hadOverdueInvoiceInPreviousContracts": "flag_teve_boleto_atrasado__contratos_anteriores",
}

In [19]:
vars_blend4 = ['score_proposto__bvs', 'SERASA_CHSV5', 'age', 'property_type', 'qtde_restricoes__consulta_realizada', 'rental_value', 'income', 'agency_pc4_mais_100_contratos__pc_categorias', 'city_pc4_mais_100_contratos__pc_categorias', 'flag_tem__contratos_anteriores', 'flag_teve_boleto_atrasado__contratos_anteriores']

id_vars = ['contract_id', 'requested_at', 'CPF_CNPJ', 'message_decisao', 'message_blendRegressaoPredict']

In [20]:
df_predict = (cp_final_df.copy()).rename(columns=rename_dict)
df_predict["REGRA_BLEND_4"] = np.where(
    df_predict["score_proposto__bvs"] < 334,
    "E_BVS",
    "BLEND4",
)

df_predict["SCORE_BVS"] = df_predict["score_proposto__bvs"]
df_predict["SCORE_SERASA"] = df_predict["SERASA_CHSV5"]
df_predict["RENDA"] = df_predict["income"]

df_predict.head()

,CPF_CNPJ,contract_id,requested_at,data_ultima_consulta,qtd_proponentes,score_imobiliaria,model,id_consulta,id_funcionalidade,documento,...,request_blend3Variables_hadOverdueInvoiceInPreviousContracts,request_blend3Variables_cityPc4HistoryOver100Contracts,request_blend3Variables_realEstatePc4HistoryOver100Contracts,qtde_pefin,valor_pefin_total,modalidades_pefin,REGRA_BLEND_4,SCORE_BVS,SCORE_SERASA,RENDA
0,33105054804,4152079,2026-06-18,2026-06-18,1,NI,BLEND3_3,5971967,32,33105054804,...,NaN,NaN,NaN,NaN,NaN,NaN,BLEND4,NaN,NaN,6850.0
1,7691151527,4156419,2026-06-16,2026-06-16,1,NI,BLEND3_3,5956396,34,07691151527,...,NaN,NaN,NaN,2.0,275.0,"CREDIARIO,FINANCIAMENT",BLEND4,NaN,NaN,3288.0
2,40664954863,4159085,2026-06-18,2026-06-18,1,E,BLEND3_3,5967580,34,40664954863,...,NaN,NaN,NaN,NaN,NaN,NaN,BLEND4,NaN,NaN,1849.5
3,12490765957,4165665,2026-06-12,2026-06-20,1,NI,BLEND3_3,5978513,34,12490765957,...,NaN,NaN,NaN,NaN,NaN,NaN,BLEND4,NaN,NaN,1918.0
4,12010654536,4167478,2026-06-15,2026-06-15,1,NI,BLEND3_3,5949536,34,12010654536,...,NaN,NaN,NaN,NaN,NaN,NaN,BLEND4,NaN,NaN,1575.5


In [21]:
df_predict.groupby("REGRA_BLEND_4", dropna=False).size()

REGRA_BLEND_4
BLEND4    111839
E_BVS        156
dtype: int64

In [22]:
# df_predict.to_csv(ANALYTICS_DIR/"df_predict_blend4_pre.csv", index=False)

## Criacão das Variáveis

In [23]:
df_predict_vars = prepare_blend4_variables(df_predict)
df_predict_escorada = predict_blend4_1(df_predict_vars)

## Criar Rating Blend4

In [24]:
score = pd.to_numeric(df_predict_escorada["pred_blend4_1_to_score"], errors="coerce")

conditions = [
    score.between(480, 1000),
    score.between(443, 479),
    score.between(408, 442),
    score.between(343, 407),
    score.between(0, 342),
]

choices = ["A", "B", "C", "D", "E"]

df_predict_escorada["rating_manual_blend4"] = np.select(conditions, choices, default=None)

In [25]:
score = pd.to_numeric(df_predict_escorada["message_blendRegressaoPredict"], errors="coerce")

conditions = [
    score.between(480, 1000),
    score.between(443, 479),
    score.between(408, 442),
    score.between(343, 407),
    score.between(0, 342),
]

choices = ["A", "B", "C", "D", "E"]

df_predict_escorada["rating_json_blend4"] = np.select(conditions, choices, default=None)

## Salvar Base como se fosse Blend4

In [26]:
# df_predict_blend4 = df_predict_escorada[df_predict_escorada["message_decisao"] == "BLEND3_3"].copy()
# df_predict_blend4["message_decisao"] = df_predict_blend4["message_decisao"].replace("BLEND3_3", "BLEND4")

# cp_final_df = pd.concat([df_predict_escorada, df_predict_blend4])
cp_final_df = df_predict_escorada
cp_final_df

,CPF_CNPJ,contract_id,requested_at,data_ultima_consulta,qtd_proponentes,score_imobiliaria,model,id_consulta,id_funcionalidade,documento,...,SERASA_CHSV5__normalized4_1,age__normalized4_1,qtde_restricoes__consulta_realizada__normalized4_1,income_commitment__normalized4_1,agency_pc4_mais_100_contratos__pc_categorias__normalized4_1,city_pc4_mais_100_contratos__pc_categorias__normalized4_1,pred_blend4_1,pred_blend4_1_to_score,rating_manual_blend4,rating_json_blend4
0,33105054804,4152079,2026-06-18,2026-06-18,1,NI,BLEND3_3,5971967,32,33105054804,...,0.0,0.55,0.0,-0.558874,0.000000,0.000000,0.477315,523.0,A,A
1,7691151527,4156419,2026-06-16,2026-06-16,1,NI,BLEND3_3,5956396,34,07691151527,...,0.0,-0.50,2.0,-0.277898,0.000000,0.000000,0.499599,500.0,A,B
2,40664954863,4159085,2026-06-18,2026-06-18,1,E,BLEND3_3,5967580,34,40664954863,...,0.0,0.05,0.0,0.662630,0.018516,0.625237,0.582800,417.0,C,E
3,12490765957,4165665,2026-06-12,2026-06-20,1,NI,BLEND3_3,5978513,34,12490765957,...,0.0,-0.45,0.0,10.566962,0.000000,1.464816,0.774663,225.0,E,E
4,12010654536,4167478,2026-06-15,2026-06-15,1,NI,BLEND3_3,5949536,34,12010654536,...,0.0,-0.75,0.0,0.452325,0.000000,0.000000,0.501865,498.0,A,A
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
111990,8227376983,4345686,2026-07-07,2026-07-07,1,D,BLEND3_3,6062771,33,08227376983,...,0.0,0.00,3.0,2.555369,0.000000,1.246605,0.633352,367.0,D,E
111991,48788388840,4345766,2026-07-07,2026-07-07,1,F,BLEND3_3,6062870,33,48788388840,...,0.0,-0.65,3.0,0.443876,0.000000,0.124206,0.548543,451.0,B,E
111992,47313079885,4345814,2026-07-07,2026-07-07,1,A,BLEND3_3,6063099,32,47313079885,...,0.0,-0.30,0.0,1.243070,0.000000,-0.501770,0.522669,477.0,B,A
111993,62043840312,4345874,2026-07-07,2026-07-07,1,NI,BLEND3_3,6063022,34,62043840312,...,0.0,-0.40,0.0,0.227300,0.000000,0.768522,0.543542,456.0,B,C


In [27]:
bvs = pd.to_numeric(cp_final_df["SCORE_BVS"], errors="coerce")
score = pd.to_numeric(cp_final_df["pred_blend4_1_to_score"], errors="coerce")

conditions = [
    bvs <= 334,                         # corte customizado BVS → E
    score.between(763, 1000),           # 763 – 1000
    score.between(704, 762),            # 704 – 762
    score.between(653, 703),            # 653 – 703
    score.between(607, 652),            # 607 – 652
    score.between(562, 606),            # 562 – 606
    score.between(520, 561),            # 520 – 561
    score.between(480, 519),            # 480 – 519
    score.between(443, 479),            # 443 – 479
    score.between(408, 442),            # 408 – 442
    score.between(375, 407),            # 375 – 407
    score.between(343, 374),            # 343 – 374
    score.between(307, 342),            # 307 – 342
    score.between(0, 306),              # 0 – 306
]

choices = [
    "E.E",      # override BVS ≤ 334
    "1.A+",
    "2.A",
    "3.A",
    "4.B+",
    "5.B+",
    "6.B",
    "7.B",
    "8.C",
    "9.D+",
    "10.D",
    "11.D",
    "E-1.E",
    "E-2.E",
]

cp_final_df["rating_cl_pol_blend4"] = np.select(conditions, choices, default=None)

## Salvar

In [28]:
cp_final_df.groupby("REGRA_BLEND_4", dropna=False).size()

REGRA_BLEND_4
BLEND4    111839
E_BVS        156
dtype: int64

In [29]:
cp_final_df.groupby("message_decisao", dropna=False).size()

message_decisao
                           11
BLEND3_3                95712
BLEND_4                  1613
BLEND_REGRESSAO_2026     7799
BVS_CUSTOM               1628
BVS_CUSTOM_V2              39
HFT1                        6
HVA3                     4475
HVA4                      712
dtype: int64

In [30]:
cp_final_df.groupby("model", dropna=False).size()

model
                           11
BLEND3_3                95712
BLEND_4                  1613
BLEND_REGRESSAO_2026     7799
BVS_CUSTOM               1628
BVS_CUSTOM_V2              39
HFT1                        6
HVA3                     4475
HVA4                      712
dtype: int64

In [31]:
cp_final_df.groupby("REGRA_BLEND_4", dropna=False).size()

REGRA_BLEND_4
BLEND4    111839
E_BVS        156
dtype: int64

In [32]:
cp_final_df.to_csv(ANALYTICS_DIR/"df_predict_blend4.csv", index=False)

## Investigação Sexta-Feira (03/07)

In [33]:
cp_final_df[
    (pd.to_datetime(cp_final_df["requested_at"]).dt.normalize() == "2026-07-03")
    & (cp_final_df["message_decisao"] == "BLEND_4")
    & (cp_final_df["pred_blend4_1_to_score"] != cp_final_df["message_blendRegressaoPredict"])
][id_vars + vars_blend4 + ["pred_blend4_1_to_score", "message_blendRegressaoPredict"]]

,contract_id,requested_at,CPF_CNPJ,message_decisao,message_blendRegressaoPredict,score_proposto__bvs,SERASA_CHSV5,age,property_type,qtde_restricoes__consulta_realizada,rental_value,income,agency_pc4_mais_100_contratos__pc_categorias,city_pc4_mais_100_contratos__pc_categorias,flag_tem__contratos_anteriores,flag_teve_boleto_atrasado__contratos_anteriores,pred_blend4_1_to_score,message_blendRegressaoPredict
950,4331023,2026-07-03,1615508945,BLEND_4,421.0,409.0,447.0,64.0,1,0,3800.0,15275.5,0.00000,0.104675,0.0,0.0,388.0,421.0
952,4331076,2026-07-03,7569622936,BLEND_4,655.0,693.0,560.0,33.0,1,2,2300.0,21509.0,0.00000,0.130864,1.0,0.0,625.0,655.0
2049,4331863,2026-07-03,41262158672,BLEND_4,510.0,546.0,538.0,63.0,0,0,7100.0,5822.5,0.00000,0.000000,0.0,0.0,552.0,510.0
2051,4332148,2026-07-03,40055469949,BLEND_4,607.0,588.0,648.0,65.0,1,0,1950.0,8220.0,0.00000,0.140461,0.0,0.0,574.0,607.0
4165,4329847,2026-07-03,85112364653,BLEND_4,239.0,419.0,245.0,59.0,1,6,3800.0,5959.5,0.00000,0.172243,0.0,0.0,216.0,239.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
108508,4332548,2026-07-03,48859970890,BLEND_4,NaN,248.0,468.0,34.0,1,0,1400.0,0.0,0.12931,0.102330,0.0,0.0,328.0,NaN
110754,4331066,2026-07-03,17354540609,BLEND_4,316.0,415.0,368.0,21.0,0,0,5000.0,2671.5,0.00000,0.076716,0.0,0.0,287.0,316.0
110761,4331831,2026-07-03,2210714079,BLEND_4,821.0,775.0,842.0,37.0,1,0,2000.0,7672.0,0.00000,0.087567,0.0,0.0,800.0,821.0
111848,4329734,2026-07-03,12713468779,BLEND_4,493.0,453.0,463.0,30.0,1,0,1900.0,4521.0,0.00000,0.058575,0.0,0.0,460.0,493.0
